# Lecture 19 (Linear programming)

## Exercise 19.1 (linear program)
<!-- taken from www-sop.inria.fr/members/Nicolas.Nisse/lectures/5LinearProg.pdf -->

Solve the below linear program using `scipy.optimize.linprog`.

Maximize (objective function)

> 350·_x_<sub>1</sub> + 300·_x_<sub>2</sub>

subject to

> _x_<sub>1</sub> + _x_<sub>2</sub> ≤ 200  
> 9·_x_<sub>1</sub> + 6·_x_<sub>2</sub> ≤ 1566  
> 12·_x_<sub>1</sub> + 16·_x_<sub>2</sub> ≤ 2880  
> _x_<sub>1</sub> ≥ 3  
> _x_<sub>2</sub> ≥ 7

_Hint_. The maximum of the objective function is 66100.

In [1]:
from scipy.optimize import linprog
import numpy as np

c = np.array([350, 300])

A_ub = np.array([[ 1,  1],
                 [ 9,  6],
                 [12, 16],
                 [-1,  0],
                 [ 0, -1]])

b_ub = np.array([200, 1566, 2880, -3, -7])

# negates c to make maximization to a minimization problem
result = linprog(-c, A_ub=A_ub, b_ub=b_ub)

print(result)

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -66100.0
              x: [ 1.220e+02  7.800e+01]
            nit: 2
          lower:  residual: [ 1.220e+02  7.800e+01]
                 marginals: [ 0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  0.000e+00  1.680e+02  1.190e+02
                              7.100e+01]
                 marginals: [-2.000e+02 -1.667e+01 -0.000e+00 -0.000e+00
                             -0.000e+00]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0


## Exercise 19.2 - handin 9 (maximum flow)

![Max flows](images/max-flow.png)

In the lecture we solved a _maximum flow_ problem using `scipy.optimize.linprog` for a concrete graph with edge capacities. The graph is showing to the right. Red numbers are the edge numbers, and blue numbers are the edge capacities.

1.  Assume we represent the `edges` of the graph by a list of triples, each triple representing an edge (_from_, _to_, _edge capacity_). Together with the names of the `source` and `sink` nodes, we have a complete representation of a flow problem. The below is representation of the graph from the lecture.

        edges = [
            ('A', 'C', 4),
            ('A', 'B', 3),
            ('C', 'E', 1),
            ('C', 'D', 1),
            ('B', 'E', 3),
            ('B', 'D', 1),
            ('E', 'D', 3),
            ('E', 'F', 1),
            ('D', 'F', 5)
        ]

        source = 'A'
        sink = 'F'

    Write a program that, given any flow problem in the above representation,  finds the maximum flow from `source` to `sink` in the graph.

    _Hint_. Convert the input to the representation used in the lecture.


2.  We now want to consider a variant of the flow problem. Assume we are given a value `flow` for the flow that we want to send from the `source` to the `sink` (not necessarily the maximum flow), but now each edge also has a _cost_ for sending one unit of flow along the edge. The input consist of a list of edges represented by 4-tuples (_from_, _to_, _capacity_, _cost_), and the `source`, `sink` and `flow` values. See example below, where the cost is 2 for sending one unit of flow from 'D' to 'F' and it is for free to send one unit of flow from 'E' and 'D'.

        edges = [
            ('A', 'C', 4, 1),
            ('A', 'B', 3, 1),
            ('C', 'E', 1, 1),
            ('C', 'D', 1, 1),
            ('B', 'E', 3, 1),
            ('B', 'D', 1, 1),
            ('E', 'D', 3, 0),
            ('E', 'F', 1, 1),
            ('D', 'F', 5, 2)
        ]

        source = 'A'
        sink = 'F'
        flow = 4

    Implement an algorithm that computes a flow from `source` to `sink` of size `flow` that has a total minimum cost.

    _Hint_. Add constraints such that (1) the flow must be equal to `flow`, (2) the subject is now to minimize the total cost of the flow, where the total cost is the sum over all edges of the flow along the edge multiplied by the edge cost.

    _Example_. There exist a solution to the above flow problem with total cost 15.

In [30]:
import numpy as np
from scipy.optimize import linprog

edges = [
    ('A', 'B', 3),
    ('A', 'C', 4),
    ('B', 'D', 1),
    ('B', 'E', 3),
    ('C', 'D', 1),
    ('C', 'E', 1),
    ('D', 'F', 5),
    ('E', 'D', 3),
    ('E', 'F', 1)
]

source = 'A'
sink = 'F'

edge_from, edge_to, capacity = zip(*edges)

nodes = list({*edge_from, *edge_to} - {source, sink})

sinks = np.array([int(t == sink) for t in edge_to])

conservation = np.array([[(i == v) - (i == u) for u, v, _ in edges] for i in nodes])

res = linprog(
    -sinks,
    A_eq=conservation,
    b_eq=np.zeros(len(nodes)),
    A_ub=np.eye(len(edges)),
    b_ub=capacity
)

print('Edge flows:', res.x)
print('Flow value:', -res.fun)

Edge flows: [3. 2. 1. 2. 1. 1. 4. 2. 1.]
Flow value: 5.0


In [31]:
from scipy.optimize import linprog

edges = [
    ('A', 'B', 3, 1),
    ('A', 'C', 4, 1),
    ('B', 'D', 1, 1),
    ('B', 'E', 3, 1),
    ('C', 'D', 1, 1),
    ('C', 'E', 1, 1),
    ('D', 'F', 5, 2),
    ('E', 'D', 3, 0),
    ('E', 'F', 1, 1)
]

source = 'A'
sink = 'F'
flow = 4

edge_from, edge_to, capacity, cost = zip(*edges)

nodes = list({*edge_from, *edge_to} - {source, sink})

sinks = [int(t == sink) for t in edge_to]

conservation = [[(i == v) - (i == u) for u, v, _, _ in edges] for i in nodes]

A_eq = np.array(conservation + [sinks])
b_eq = np.array([0] * len(nodes) + [flow])

res = linprog(
    cost,
    A_eq=A_eq,
    b_eq=b_eq,
    A_ub=np.eye(len(edges)),
    b_ub=capacity
)

print('Edge flows:', res.x)
print('Flow value:', res.fun)

Edge flows: [2. 2. 0. 2. 1. 1. 3. 2. 1.]
Flow value: 15.0
